Step 1: Create Dataset (sales_data.csv)

In [1]:
import pandas as pd
import os

# Ensure data folder exists
os.makedirs("data", exist_ok=True)

# Define dataset with some missing & duplicate values
data = {
    "order_id": [201, 202, 203, 204, 205, 202],
    "customer": ["Alice", "Bob", "Charlie", "David", "Eva", "Bob"],
    "region": ["North", "East", None, "West", "South", "East"],
    "revenue": [120000, 75000, 50000, None, 30000, 75000],
    "cost": [80000, 40000, 30000, 20000, 15000, 40000]
}
# Create DataFrame
df = pd.DataFrame(data)

# Save original CSV
df.to_csv("data/sales_data.csv", index=False)

print("Dataset created: data/sales_data.csv")


Dataset created: data/sales_data.csv


Step 2: Process Data

In [2]:
# Load raw dataset
df = pd.read_csv("data/sales_data.csv")

# Remove duplicates by order_id
df = df.drop_duplicates(subset=["order_id"])

# Handle missing values
df["region"] = df["region"].fillna("Unknown")
df["revenue"] = df["revenue"].fillna(0)

# Add profit_margin column
df["profit_margin"] = (df["revenue"] - df["cost"]) / df["revenue"]
df["profit_margin"] = df["profit_margin"].fillna(0)

# Categorize customer segments
def segment_customer(revenue):
    if revenue > 100000:
        return "Platinum"
    elif 50000 < revenue <= 100000:
        return "Gold"
    else:
        return "Standard"

df["customer_segment"] = df["revenue"].apply(segment_customer)


Step 3: Save Outputs

In [3]:
# Save raw file (unchanged)
raw_df = pd.read_csv("data/sales_data.csv")
raw_df.to_csv("data/raw_sales_data.csv", index=False)

# Save processed file
df.to_csv("data/processed_sales_data.csv", index=False)

print("Files saved in 'data/' folder:")
print("- raw_sales_data.csv (unchanged)")
print("- processed_sales_data.csv (cleaned + enriched)")


Files saved in 'data/' folder:
- raw_sales_data.csv (unchanged)
- processed_sales_data.csv (cleaned + enriched)


In [5]:
!pip install azure-storage-blob


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 18.4 MB/s eta 0:00:00


Task 2 — Azure Blob Storage Integration

In [ ]:
import os
from azure.storage.blob import BlobServiceClient

# Read environment variables (set in Azure DevOps → Pipeline → Variables)
account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY")
container_name = os.getenv("AZURE_CONTAINER_NAME")

# Connect to Azure Blob Service
connection_string = f"DefaultEndpointsProtocol=https;AccountName={account_name};AccountKey={account_key};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Get container client
container_client = blob_service_client.get_container_client(container_name)

# Files to upload
files_to_upload = ["data/raw_sales_data.csv", "data/processed_sales_data.csv"]

for file_path in files_to_upload:
    blob_name = os.path.basename(file_path)  # file name only
    with open(file_path, "rb") as data:
        container_client.upload_blob(name=blob_name, data=data, overwrite=True)
        print(f"Uploaded {blob_name} to Azure Blob Storage in container '{container_name}'.")
